# មេរៀនទី 10 - ភ្នាក់ងារ AI ក្នុងការផលិត

នៅក្នុងមេរៀននេះ អ្នកនឹងរៀនអំពី **លំនាំក្នុងការផលិត** សម្រាប់ភ្នាក់ងារ AI ដោយប្រើ Microsoft Agent Framework (Python). We cover:

- **ភាពអាចតាមដានបាន** — បន្ថែមការវាស់ពេល និងកំណត់ហេតុនៅលើអន្តរកម្មរវាងភ្នាក់ងារ
- **ការវាយតម្លៃ** — ប្រើភ្នាក់ងារវាយតម្លៃដើម្បីផ្តល់ពិន្ទុទៅលើគុណភាពនៃការឆ្លើយតប
- **ការគ្រប់គ្រងថ្លៃ** — យុទ្ធសាស្រ្តសម្រាប់បង្កើនប្រសិទ្ធភាព token និងការជ្រើសម៉ូឌែល

ស្ថានភាពនេះគឺជា **ភ្នាក់ងារធ្វើដំណើរ** ដែលជួយអ្នកប្រើរៀបចំដំណើរ រួមទាំងការតាមដាន និងការវាយតម្លៃដែលបានដាក់បន្ថែម


## ការដំឡើង


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## ការពិចារណាសម្រាប់ផលិតកម្ម

ការផ្លាស់ប្តូរភ្នាក់ងារ AI ពីគំរូទៅដល់ការផលិត តម្រូវឱ្យមានការយកចិត្តទុកដាក់យ៉ាងម៉ត់ចត់ចំពោះស្តម្បាចម្បងបី៖

1. **ភាពអាចសង្កេតបាន** — អ្នកត្រូវការមើលឃើញច្បាស់អំពីអ្វីដែលភ្នាក់ងារកំពុងធ្វើ, វាចំណាយពេលប៉ុន្មាន, និងឧបករណ៍ណាខ្លះដែលវាហៅ។ បើខ្វះការតាមដាន និងកត់ត្រា ការដោះស្រាយបញ្ហាក្នុងការផលិតប្រហែលជាមិនអាចធ្វើបានឡើយ។

2. **ការវាយតម្លៃ** — ការត្រួតពិនិត្យគុណភាពដោយស្វ័យប្រវត្តិធានាថាចម្លើយរបស់ភ្នាក់ងារនៅតែមានភាពត្រឹមត្រូវ ពេញលេញ និងមានប្រយោជន៍ជាយូរអង្វែង។ ភ្នាក់ងារវាយតម្លៃអាចផ្ដល់ពិន្ទុលើចម្លើយដោយផ្អែកលើលក្ខខណ្ឌដែលបានកំណត់។

3. **ការគ្រប់គ្រងចំណាយ** — ការប្រើប្រាស់តូកិនមានឥទ្ធិពលផ្ទាល់លើចំណាយ។ យុទ្ធសាស្ត្រដូចជា ការបង្កើនប្រសិទ្ធភាពពាក្យបញ្ជា ការជ្រើសរើសម៉ូដែល និងការផ្ទុកបន្ទុក (caching) ជួយរក្សាចំណាយឲ្យក្រោមការគ្រប់គ្រងដោយមិនប៉ះពាល់គុណភាព។


## ការសាងសង់ភ្នាក់ងារដែលអាចតាមដានបាន

យើងកំណត់ឧបករណ៍សម្រាប់ធ្វើដំណើរ ហើយបំពាក់ការហៅភ្នាក់ងារជាមួយការវាស់ពេល ដើម្បីយើងអាចតាមដានការពន្យារពេល។ នៅក្នុងបរិបទផលិត (production) អ្នកនឹងបញ្ចូលជាមួយ OpenTelemetry ឬ backend តាមដានស្រដៀង។


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## លំនាំនៃការវាយតម្លៃ

លំនាំទូទៅនៅក្នុងការផលិត គឺប្រើភ្នាក់ងារទីពីរជា **អ្នកវាយតម្លៃ**។ អ្នកវាយតម្លៃនឹងពិន្ទុនូវចម្លើយរបស់ភ្នាក់ងារសំខាន់ទៅតាមលក្ខខណ្ឌដែលបានកំណត់ជាមុន ដូចជា ភាពពេញលេញ ភាពត្រឹមត្រូវ និងភាពមានប្រយោជន៍។

នេះអនុញ្ញាតឲ្យ៖
- ប្រព័ន្ធត្រួតពិនិត្យគុណភាពស្វ័យប្រវត្តិ មុនពេលចម្លើយទៅដល់អ្នកប្រើ
- ការរកឃើញបញ្ហាការត្រឡប់ក្រោយ (regression) ពេលដែលព័ត៌មានបញ្ជា ឬ ម៉ូឌែលផ្លាស់ប្តូរ
- ការតាមដានជាប់លាប់លើការសម្តែងរបស់ភ្នាក់ងារតាមពេលវេលា


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## យុទ្ធសាស្ត្រគ្រប់គ្រងថ្លៃចំណាយ

ការគ្រប់គ្រងថ្លៃចំណាយមានសារៈសំខាន់សម្រាប់ភ្នាក់ងារផលិតកម្ម AI។ នៅទីនេះជាវិធីសាស្ត្រសំខាន់ៗ៖

| យុទ្ធសាស្ត្រ | ការពិពណ៌នា |
|---|---|
| **បង្កើនប្រសិទ្ធភាពសំណើ** | រក្សាណែនាំប្រព័ន្ធឲ្យខ្លីសង្ខេប។ លុបបរិបទដែលមិនចាំបាច់ ដើម្បីបន្ថយចំនួន tokens នៃការបញ្ចូល។ |
| **ការជ្រើសម៉ូដែល** | ប្រើម៉ូដែលតូច និងថោកជាង (ឧទាហរណ៍ GPT-4o-mini) សម្រាប់កិច្ចការ​សាមញ្ញដូចជា ការចាត់ថ្នាក់ ឬ ការទាញយក ហើយរក្សាម៉ូដែលធំសម្រាប់ការគិតវិភាគស្មុំស្មាញ។ |
| **ការផ្ទុកចាំ** | ផ្ទុកលទ្ធផលឧបករណ៍ និងសំណើញឹកញាប់ ដើម្បីជៀសវាងការហៅ API ដែលចម្លងគ្នា។ |
| **ថវិកា tokens** | កំណត់ដែនកំណត់ `max_tokens` ដើម្បីទប់ស្កាត់ការឆ្លើយតបដែលវែងពេក។ |
| **ការបញ្ចូលជាក្រុម** | រួមបញ្ចូលសំណើអ្នកប្រើជាច្រើនទៅក្នុងការហៅ API តែមួយនៅពេលដែលអាចធ្វើបាន។ |

ក្នុងការអនុវត្ត ការទទួលយកវិធីសាស្ត្រជាចំណាត់ថ្នាក់មានប្រសិទ្ធភាព៖ បញ្ជូនសំណើងាយៗទៅម៉ូដែលលឿន និងថោក ហើយតែមើលទៅឡើងសំណើស្មុគស្មាញទៅម៉ូដែលដែលមានសមត្ថភាពខ្ពស់។


## សេចក្ដីសង្ខេប

នៅក្នុងមេរៀននេះ អ្នកបានរៀនពីរបៀប:

1. **បន្ថែមការសង្កេតមើល** ទៅលើអន្តរកម្មរបស់ភ្នាក់ងារ ដោយកត់ពេល និងកំណត់ហេតុ ដែលបង្កើតមូលដ្ឋានសម្រាប់ការតាមដាន និងការត្រួតពិនិត្យ.
2. **វាយតម្លៃចម្លើយរបស់ភ្នាក់ងារ** យ៉ាងស្វ័យប្រវត្តិដោយប្រើភ្នាក់ងារវាយតម្លៃ ដែលផ្ដល់ពិន្ទុលើភាពពេញលេញ ភាពត្រឹមត្រូវ និងភាពមានប្រយោជន៍.
3. **គ្រប់គ្រងចំណាយ** តាមរយៈការបង្កើត prompt ឲ្យមានប្រសិទ្ធភាព, ការជ្រើសរើសម៉ូឌែល, ការផ្ទុកក្នុងចាំ (caching), និងថវិកាសម្រាប់ token.

លំនាំការផលិតទាំងនេះ ជួយធានាថា ភ្នាក់ងារ AI របស់អ្នក ទុកចិត្តបាន អាចវាស់បាន និងមានប្រសិទ្ធភាពក្នុងការចំណាយនៅពេលពង្រីក.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការមិនទទួលខុសត្រូវ**:
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាបកប្រែដោយបច្ចេកវិទ្យា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខិតខំដើម្បីផ្តល់ភាពត្រឹមត្រូវ សូមជ្រាបថា ការបកប្រែដោយស្វ័យប្រវត្តិក៏អាចមានកំហុស ឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមក្នុងភាសារបស់វាគួរត្រូវបានចាត់ទុកជាប្រភពដែលមានសុពលភាព។ សម្រាប់ព័ត៌មានសំខាន់ៗ យើងសូមណែនាំឱ្យពិចារណាបកប្រែដោយអ្នកបកប្រែវិជ្ជាជីវៈ។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការពន្យល់ខុសដែលកើតឡើងពីការប្រើប្រាស់ការបកប្រែនេះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
